# Model 2: Disengagement-Risk Prediction with XGBoost
**Component**: AI-Powered Multimodal Student Digital Twin & Cognitive Memory Engine
Predicts risk of future 14-day inactivity / withdrawal with calibrated alert thresholds.

In [ ]:
# Cell 1: Install Dependencies
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib
print('Dependencies installed.')

In [ ]:
# Cell 2: Run Data Preprocessing
!python dataset_preparation/generate_sample_data.py
!python dataset_preparation/preprocess_all.py

In [ ]:
# Cell 3: Train Disengagement Model & Tune Early Warning Threshold
!python 02_disengagement_risk_model/train_disengagement.py

In [ ]:
# Cell 4: View Evaluation Charts (ROC, Precision-Recall & Risk Factors)
from IPython.display import Image, display
display(Image('reports/disengagement_model_evaluation.png'))

In [ ]:
# Cell 5: Test Risk Prediction with Calibrated Threshold
import json
import joblib
import pandas as pd

model = joblib.load('models/disengagement_xgb.joblib')
features = joblib.load('models/disengagement_features.joblib')
with open('models/disengagement_threshold.json') as f:
    meta = json.load(f)
threshold = meta['optimal_threshold']

inactive_sample = pd.DataFrame([{
    'active_days_7d': 0,
    'active_days_14d': 1,
    'clicks_7d': 0,
    'click_change': -25,
    'days_since_last_activity': 12,
    'previous_inactivity_events': 1
}])[features]

risk_prob = float(model.predict_proba(inactive_sample)[0, 1])
alert = risk_prob >= threshold
print(f'Predicted Disengagement Risk: {risk_prob:.1%}')
print(f'Early Warning Triggered:      {alert} (Threshold: {threshold:.2f})')

In [ ]:
# Cell 6: Download Artifacts for Backend Service
from google.colab import files
files.download('models/disengagement_xgb.joblib')
files.download('models/disengagement_features.joblib')
files.download('models/disengagement_threshold.json')